### **Convolutional Neural Network (CNN)**

A **Convolutional Neural Network (CNN)** is a deep learning architecture mainly used for **image processing, computer vision and pattern recognition**.

CNNs are especially powerful because they can automatically learn features such as **edges → shapes → textures → objects** from images.

---

#### **Why CNN?**

Suppose we have a `28 × 28` grayscale image. A fully connected neural network would flatten it:

```text
28 × 28 → 784 values
```

and connect every pixel to neurons. The problem is that it:

* has many parameters
* ignores spatial relationships
* is computationally expensive for larger images

CNNs solve this using **convolutional filters** that scan across the image and learn local patterns.

---

#### **CNN Architecture**

A typical CNN looks like:

```text
Input Image-> Convolution-> Activation (ReLU)-> Pooling-> Convolution-> Activation (ReLU)-> Pooling-> Flatten-> Fully Connected Layer-> Output
```

---

#### **Convolution**

The most important operation in CNN is **convolution**. A small matrix called a **kernel/filter** moves across the image. Example:

```text
Image                 Filter

1  2  3               1  0
4  5  6               0 -1
7  8  9
```

The filter performs element-wise multiplication and sums the results.

```text
(1×1) + (2×0)
+ (4×0) + (5×-1)

= 1 - 5
= -4
```

The filter continues moving across the image and produces a **feature map**.

---

#### **`nn.Conv2d()` in PyTorch**

PyTorch provides convolution through **`nn.Conv2d()`**. Basic syntax:

```python
nn.Conv2d(
    in_channels,
    out_channels,
    kernel_size
)
```

Example:

```python
import torch
import torch.nn as nn

conv = nn.Conv2d(
    in_channels=1,
    out_channels=32,
    kernel_size=3
)
```

Meaning:

```text
Input channels  = 1
Output channels = 32
Kernel size     = 3×3
```

So the network learns **32 different 3×3 filters**.

---

#### **Input Shape in CNN**

PyTorch expects image tensors in this format: **`(batch_size, channels, height, width)`**. For example:

```python
X = torch.randn(32, 1, 28, 28)
```

means:

```text
32  → batch size
1   → grayscale channel
28  → height
28  → width
```

For an RGB image: **`X = torch.randn(32, 3, 224, 224)`** because RGB has: **`3 channels`**.

---

#### **ReLU Activation**

After convolution, we normally apply **ReLU**. Mathematically: **`ReLU(x) = max(0, x)`**. Example:

```text
Input:

-5   2
 3  -1

After ReLU:

0   2
3   0
```

ReLU introduces **non-linearity** allowing the network to learn complex patterns.

---

#### **Pooling**

Pooling reduces the spatial dimensions of feature maps. The most common type is **Max Pooling**.

```python
nn.MaxPool2d(kernel_size=2)
```

Example:

```text
2  5
7  3
```

Max pooling produces **7** because **max(2, 5, 7, 3) = 7**. A `2×2` max pooling layer usually reduces **28 × 28** to **14 × 14**.

---

#### **Flatten**

After convolution and pooling, we need to convert the feature maps into a 1D vector before passing them to a fully connected layer.

```python
nn.Flatten()
```

For example: **7 × 7 × 64** becomes **3136** because **7 × 7 × 64 = 3136**.

---

#### **Complete CNN Using PyTorch**

Here is a simple CNN for **MNIST digit classification**.

In [1]:
import torch
import torch.nn as nn

class CNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            # Convolutional Layer
            nn.Conv2d(
                in_channels = 1,
                out_channels = 32,
                kernel_size = 3,
                padding = 1
            ),

            # Activation
            nn.ReLU(),

            # Pooling 
            nn.MaxPool2d(kernel_size = 2),

            # Second convolution
            nn.Conv2d(
                in_channels = 32,
                out_channels = 64,
                kernel_size = 3,
                padding = 1
            ),

            # Activation
            nn.ReLU(),

            # Pooling
            nn.MaxPool2d(kernel_size = 2),

            # Flatten
            nn.Flatten(),

            # Fully connected layer
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),

            # Output layer
            nn.Linear(128, 10)
        )

    def forward(self, x):
        return self.network(x)

---

#### **Understanding the Dimensions**

Let's follow an MNIST image through the network. 

* Initial image: `1 × 28 × 28`
* After first convolution: `32 × 28 × 28` Because `out_channels = 32 and padding = 1`.
* Then MaxPool: `32 × 14 × 14`
* Second convolution: `64 × 14 × 14`
* Second MaxPool: `64 × 7 × 7`
* Flatten: `64 × 7 × 7 = 3136`. Then `3136-> 128-> 10`

The `10` outputs represent: `0 1 2 3 4 5 6 7 8 9 `


---

#### **Create the Model**

In [2]:
model = CNN()
print(model)

CNN(
  (network): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Flatten(start_dim=1, end_dim=-1)
    (7): Linear(in_features=3136, out_features=128, bias=True)
    (8): ReLU()
    (9): Linear(in_features=128, out_features=10, bias=True)
  )
)


In [3]:
# You can also test it with random data:
X = torch.randn(32, 1, 28, 28)
output = model(X)
print(output.shape)

torch.Size([32, 10])


Meaning:

```text
32 images
10 class scores per image
```

---

#### **Loss Function**

For multi-class classification, we can use:

```python
criterion = nn.CrossEntropyLoss()
```

Example: `loss = criterion(output, y)` where **y** contains class labels such as `[2, 5, 1, 9, 0, ...]`

---

#### **Optimizer**

A common optimizer is Adam:

In [5]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr = 0.001
)

---

#### **CNN Training Loop**

```python
for epoch in range(10):

    model.train()
    for images, labels in train_loader:
        # Forward pass
        outputs = model(images)

        # Calculate loss
        loss = criterion(outputs, labels)

        # Clear old gradients
        optimizer.zero_grad()

        # Backpropagation
        loss.backward()

        # Update weights
        optimizer.step()

    print(
        f"Epoch [{epoch+1}/10], "
        f"Loss: {loss.item():.4f}"
    )
```

---

#### **Important CNN Parameters**

**`kernel_size`:** Controls the size of the filter. `nn.Conv2d(1, 32, kernel_size=3)` means `3 × 3 filter`. Common choices:

```text
3 × 3
5 × 5
7 × 7
```

---

**`stride`:** Controls how far the filter moves.

```python
nn.Conv2d(
    1,
    32,
    kernel_size=3,
    stride=2
)
```

Larger stride generally produces smaller feature maps.

---

**`padding`:** Adds zeros around the image.

```python
nn.Conv2d(
    1,
    32,
    kernel_size=3,
    padding=1
)
```

Padding helps preserve spatial dimensions.

---

**`out_channels`:** Controls how many filters are learned.

```python
nn.Conv2d(
    1,
    32,
    kernel_size=3
)
```

means `32 filters`. Therefore **output channels = 32**.

---

#### **CNN vs ANN**

| Feature             | ANN                     | CNN                  |
| ------------------- | ----------------------- | -------------------- |
| Input               | Flattened               | Image/Grid           |
| Spatial information | Poor                    | Excellent            |
| Parameters          | Usually more            | Usually fewer        |
| Feature extraction  | Manual/less specialized | Automatic            |
| Image processing    | Less effective          | Very effective       |
| Main layers         | Linear                  | Conv + Pool + Linear |

---

#### **What Does a CNN Learn?**

A CNN typically learns features hierarchically:

```text
                    Image
                      ↓
              ┌─────────────┐
              │ First Layer  │
              └─────────────┘
                      ↓
                    Edges
                 /         \
            Vertical   Horizontal
               ↓           ↓
              ┌─────────────┐
              │ Second Layer│
              └─────────────┘
                      ↓
                   Shapes
                      ↓
              ┌─────────────┐
              │ Deep Layers │
              └─────────────┘
                      ↓
                Complex Objects
```
---

#### **CNN Applications**

* 🖼️ Image classification
* 👤 Face recognition
* 🚗 Autonomous driving
* 🔍 Object detection
* 🩻 Medical image analysis
* ✍️ Handwritten digit recognition
* 📹 Video analysis
* 🛰️ Satellite image analysis
* 📦 Image segmentation

---